
# AI/ML Hangman — Training Notebook (WordNet Edition)

This notebook builds a **real machine-learning model** that plays Hangman
intelligently: given the current masked word and the letters guessed so far,
it predicts which unguessed letter is most likely to appear next.

**Corpus:** built entirely from `nltk.corpus.wordnet` — no Kaggle or external
word lists.

**Pipeline:**
1. Build & filter the WordNet vocabulary
2. Compute letter-frequency statistics (global / by length / by position / co-occurrence)
3. Simulate thousands of realistic mid-game Hangman states from the vocabulary
4. Turn each state + candidate letter into a feature vector
5. Train a `HistGradientBoostingClassifier` to predict "is this letter in the word?"
6. Evaluate it against a frequency-only baseline
7. Save the trained model + statistics as artifacts for the Gradio app (`app.py`)

The exact same `corpus_utils.py` / `features.py` modules produced here are
imported by `app.py`, so training and deployment never drift apart.


## 1. Setup

In [1]:
import sys
import subprocess

def pip_install(*packages):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *packages
    ])

pip_install(
    "nltk",
    "scikit-learn",
    "pandas",
    "numpy",
    "joblib",
    "gradio"
)

print("Packages ready.")

Packages ready.


In [2]:

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
print("WordNet ready.")


WordNet ready.


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\tuhin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\tuhin\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!



# Build the WordNet vocabulary



## Create the file and imports -

In [3]:
# Create the file and imports -

import os
import string
import pandas as pd

# Min number of word in letter -
MIN_LEN = 4

# Max number of word in letter
MAX_LEN = 15 
ALPHABET = list(string.ascii_lowercase)

## WordNet setup -

In [4]:
%%writefile -a corpus_utils.py

def ensure_wordnet():
    import nltk

    try:
        nltk.data.find("corpora/wordnet")
    except LookupError:
        nltk.download("wordnet")

    try:
        nltk.data.find("corpora/omw-1.4")
    except LookupError:
        nltk.download("omw-1.4")

Appending to corpus_utils.py


## Build vocabulary -

In [5]:
%%writefile -a corpus_utils.py

def build_vocab_from_wordnet():
    ensure_wordnet()

    from nltk.corpus import wordnet as wn

    words = set()

    for synset in wn.all_synsets():
        for lemma in synset.lemmas():
            word = lemma.name().lower()

            if word.isalpha():
                words.add(word)

    total_raw = len(words)

    filtered = set()

    for word in words:
        if not word.isascii():
            continue

        if MIN_LEN <= len(word) <= MAX_LEN:
            filtered.add(word)

    vocab = sorted(filtered)

    return vocab, total_raw

Appending to corpus_utils.py


## Save and load vocabulary -

In [6]:
%%writefile -a corpus_utils.py

def save_vocab_csv(vocab, path = "wordnet_corpus.csv"):
    df = pd.DataFrame({
        "word" : vocab,
        "length" : [len(word) for word in vocab]
    })

    df.to_csv(path, index = False)

    return path


def load_or_build_vocab(csv_path="wordnet_corpus.csv"):

    if os.path.exists(csv_path):

        df = pd.read_csv(
            csv_path,
            keep_default_na=False,
            na_filter=False,
            dtype=str
        )

        vocab = sorted(df["word"].astype(str).tolist())

        return vocab

    vocab, _ = build_vocab_from_wordnet()

    save_vocab_csv(vocab, csv_path)

    return vocab

Appending to corpus_utils.py


## Build letter statistics -

In [7]:
%%writefile -a corpus_utils.py

def build_letter_stats(vocab):

    global_count = {
        c: 0 for c in ALPHABET
    }

    length_count = {}
    position_count = {}

    pair_count = {
        c: {c2: 0 for c2 in ALPHABET}
        for c in ALPHABET
    }

    length_totals = {}

    n_words = len(vocab)

    for word in vocab:

        L = len(word)

        length_totals[L] = length_totals.get(L, 0) + 1

        letters = set(word)

        # Global frequency - 
        for c in letters:
            global_count[c] += 1

        # Length-based frequency - 
        if L not in length_count:
            length_count[L] = {
                c: 0 for c in ALPHABET
            }

        for c in letters:
            length_count[L][c] += 1

        # Position frequency - 
        for pos, ch in enumerate(word):

            key = (L, pos)

            if key not in position_count:
                position_count[key] = {
                    c: 0 for c in ALPHABET
                }

            position_count[key][ch] += 1

        # Pair frequency - 
        for c1 in letters:
            for c2 in letters:

                if c1 != c2:
                    pair_count[c1][c2] += 1

Appending to corpus_utils.py


## Calculate frequencies -

In [8]:
%%writefile -a corpus_utils.py

    # Global probability - 
    global_freq = {
        c: global_count[c] / n_words
        for c in ALPHABET
    }

    # Length probability - 
    length_freq = {}

    for L, counts in length_count.items():

        total = length_totals[L]

        length_freq[L] = {
            c: counts[c] / total
            for c in ALPHABET
        }

    # Position probability - 
    position_freq = {}

    for key, counts in position_count.items():

        L, pos = key
        total = length_totals[L]

        position_freq[key] = {
            c: counts[c] / total
            for c in ALPHABET
        }

    # Pair probability -
    pair_freq = {}

    for c1 in ALPHABET:

        denominator = global_count[c1]

        if denominator == 0:
            denominator = 1

        pair_freq[c1] = {
            c2: pair_count[c1][c2] / denominator
            for c2 in ALPHABET
        }

    return {
        "global_freq": global_freq,
        "length_freq": length_freq,
        "position_freq": position_freq,
        "pair_freq": pair_freq,
        "length_totals": length_totals,
        "n_words": n_words
    }

Appending to corpus_utils.py


## Test the file - 

In [9]:
import corpus_utils

vocab = corpus_utils.load_or_build_vocab()

print("Vocabulary size:", len(vocab))
print("First 10 words:", vocab[:10])

stats = corpus_utils.build_letter_stats(vocab)

print("Number of words:", stats["n_words"])
print("Frequency of 'e':", stats["global_freq"]["e"])

Vocabulary size: 75018
First 10 words: ['aachen', 'aalborg', 'aalii', 'aalst', 'aalto', 'aardvark', 'aardwolf', 'aare', 'aarhus', 'aaron']
Number of words: 75018
Frequency of 'e': 0.639059958943187


In [10]:
from corpus_utils import build_vocab_from_wordnet, save_vocab_csv, build_letter_stats

# Build vocabulary from WordNet - 
vocab, total_raw = build_vocab_from_wordnet()
 
print("Total WordNet words:", total_raw)
print("Words after filtering:", len(vocab))

print("Minimum word length:", min(len(w) for w in vocab))
print("Maximum word length:", max(len(w) for w in vocab))

print("Sample words:", vocab[:15])
save_vocab_csv(vocab, "wordnet_corpus.csv")

print("Saved wordnet_corpus.csv with", len(vocab), "words.")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\tuhin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\tuhin\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Total WordNet words: 77503
Words after filtering: 75018
Minimum word length: 4
Maximum word length: 15
Sample words: ['aachen', 'aalborg', 'aalii', 'aalst', 'aalto', 'aardvark', 'aardwolf', 'aare', 'aarhus', 'aaron', 'aarp', 'aave', 'abaca', 'abacinate', 'aback']
Saved wordnet_corpus.csv with 75018 words.



> **Note:** `wordnet_corpus.csv` only stores the *filtered* vocabulary (words +
> lengths) derived from WordNet — it is a cache, not a bundled copy of
> WordNet itself. If it's missing, `load_or_build_vocab()` regenerates it
> automatically straight from `nltk.corpus.wordnet`.



## Letter-frequency statistics (feature building blocks) -

Before we can train a classifier we need numeric signals about English
letter patterns. We compute:

- **Global frequency** — how many words contain each letter
- **Length-conditioned frequency** — letter frequency given the secret word's length
- **Positional frequency** — how likely a letter is at a specific position, given word length
- **Co-occurrence** — how often a letter appears in words that already contain a known letter



In [11]:
import pickle
import os

stats = build_letter_stats(vocab)

os.makedirs("artifacts", exist_ok=True)

with open("artifacts/stats.pkl", "wb") as f:
    pickle.dump(stats, f)

print("Top 10 frequent letters:")

for letter, freq in sorted(
    stats["global_freq"].items(),
    key=lambda x: -x[1]
)[:10]:
    print(f"{letter.upper()}: {freq:.3f}")

Top 10 frequent letters:
E: 0.639
A: 0.593
I: 0.569
R: 0.506
N: 0.477
T: 0.462
O: 0.459
S: 0.418
L: 0.408
C: 0.330



# Feature engineering + training-data simulation -



## Imports and make_features -

In [12]:
%%writefile features.py

import random
import string

ALPHABET = list(string.ascii_lowercase)

def make_features(word_length, pattern, guessed_letters, candidate, stats):
    global_freq = stats["global_freq"]
    length_freq = stats["length_freq"].get(word_length, {})
    position_freq = stats["position_freq"]
    pair_freq = stats["pair_freq"]

    n_blanks = pattern.count("_")
    n_revealed = word_length - n_blanks
    n_guessed = len(guessed_letters)

    f_global = global_freq.get(candidate, 0.0)
    f_length = length_freq.get(candidate, 0.0)

    blank_positions = [i for i, c in enumerate(pattern) if c == "_"]

    if blank_positions:
        pos_scores = []

        for pos in blank_positions:
            key = (word_length, pos)
            pos_scores.append(
                position_freq.get(key, {}).get(candidate, 0.0)
            )

        f_position_avg = sum(pos_scores) / len(pos_scores)
        f_position_max = max(pos_scores)
    else:
        f_position_avg = 0.0
        f_position_max = 0.0

    revealed_letters = set(c for c in pattern if c != "_")

    if revealed_letters:
        pair_scores = [
            pair_freq.get(r, {}).get(candidate, 0.0)
            for r in revealed_letters
        ]
        f_pair_avg = sum(pair_scores) / len(pair_scores)
    else:
        f_pair_avg = f_global

    wrong_letters = guessed_letters - revealed_letters

    if wrong_letters:
        wrong_scores = [
            pair_freq.get(w, {}).get(candidate, 0.0)
            for w in wrong_letters
        ]
        f_wrong_pair_avg = sum(wrong_scores) / len(wrong_scores)
    else:
        f_wrong_pair_avg = f_global

    progress = n_revealed / word_length if word_length else 0.0

    return [
        f_global,
        f_length,
        f_position_avg,
        f_position_max,
        f_pair_avg,
        f_wrong_pair_avg,
        progress,
        n_blanks / max(word_length, 1),
        n_guessed / 26.0,
        word_length / 15.0,
    ]


FEATURE_NAMES = [
    "global_freq",
    "length_freq",
    "position_freq_avg",
    "position_freq_max",
    "pair_freq_with_revealed",
    "pair_freq_with_wrong",
    "progress",
    "blank_ratio",
    "guessed_ratio",
    "norm_word_length",
]

Overwriting features.py


## Game-state simulation -

In [13]:
%%writefile -a features.py

def simulate_game_state(word, rng):
    distinct_letters = list(set(word))
    rng.shuffle(distinct_letters)

    n_reveal = rng.randint(
        0,
        max(0, len(distinct_letters) - 1)
    )

    revealed = set(distinct_letters[:n_reveal])

    pattern = [
        ch if ch in revealed else "_"
        for ch in word
    ]

    not_in_word = [
        c for c in ALPHABET
        if c not in word
    ]

    rng.shuffle(not_in_word)

    n_wrong = rng.randint(
        0,
        min(5, len(not_in_word))
    )

    wrong_guessed = set(not_in_word[:n_wrong])

    guessed_letters = revealed | wrong_guessed

    return pattern, guessed_letters

Appending to features.py


## Training data generation -

In [14]:
%%writefile -a features.py

def generate_training_data(
    vocab,
    stats,
    n_samples_per_word=3,
    candidates_per_state=6,
    seed=42
):
    rng = random.Random(seed)

    X, y = [], []

    for word in vocab:

        for _ in range(n_samples_per_word):

            pattern, guessed_letters = simulate_game_state(
                word, rng
            )

            if "_" not in pattern:
                continue

            remaining_letters = [
                c for c in ALPHABET
                if c not in guessed_letters
            ]

            if not remaining_letters:
                continue

            positive_candidates = [
                c for c in remaining_letters
                if c in word
            ]

            negative_candidates = [
                c for c in remaining_letters
                if c not in word
            ]

            rng.shuffle(positive_candidates)
            rng.shuffle(negative_candidates)

            n_pos = min(
                len(positive_candidates),
                max(1, candidates_per_state // 2)
            )

            n_neg = min(
                len(negative_candidates),
                candidates_per_state - n_pos
            )

            chosen = (
                [(c, 1) for c in positive_candidates[:n_pos]]
                + [(c, 0) for c in negative_candidates[:n_neg]]
            )

            for candidate, label in chosen:

                feats = make_features(
                    len(word),
                    pattern,
                    guessed_letters,
                    candidate,
                    stats
                )

                X.append(feats)
                y.append(label)

    return X, y

Appending to features.py


## Generate training data -

In [15]:
from features import generate_training_data, FEATURE_NAMES
import numpy as np
import time

t0 = time.time()

X, y = generate_training_data(
    vocab,
    stats,
    n_samples_per_word=3,
    candidates_per_state=6,
    seed=42
)

X = np.array(X, dtype="float32")
y = np.array(y, dtype="int8")

print(f"Generated {len(X):,} training rows in {time.time()-t0:.1f}s")
print("Feature columns:", FEATURE_NAMES)
print("Positive rate:", y.mean())

Generated 1,350,324 training rows in 36.4s
Feature columns: ['global_freq', 'length_freq', 'position_freq_avg', 'position_freq_max', 'pair_freq_with_revealed', 'pair_freq_with_wrong', 'progress', 'blank_ratio', 'guessed_ratio', 'norm_word_length']
Positive rate: 0.4230569848421564


## Save the data -

In [16]:
np.save("artifacts/X.npy", X)
np.save("artifacts/y.npy", y)

print("Training data saved.")

Training data saved.



# Train the ML model -


## Split the data -

In [17]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import time

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

## Train and evaluate - 

In [19]:
print(type(clf))
print(len(FEATURE_NAMES))
print(X_test.shape)

<class 'sklearn.ensemble._hist_gradient_boosting.gradient_boosting.HistGradientBoostingClassifier'>
10
(202549, 10)


In [20]:
clf = HistGradientBoostingClassifier(
    max_iter=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

t0 = time.time()

clf.fit(X_train, y_train)

print(f"Trained in {time.time() - t0:.1f}s")

proba = clf.predict_proba(X_test)[:, 1]
pred = (proba > 0.5).astype(int)

print("Test accuracy:", accuracy_score(y_test, pred))
print("Test ROC-AUC:", roc_auc_score(y_test, proba))

Trained in 22.5s
Test accuracy: 0.7453998785479069
Test ROC-AUC: 0.8247549464170234


In [28]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    clf,
    X_test[:8000],
    y_test[:8000],
    n_repeats=3,
    random_state=42,
    n_jobs=-1
)

print("Feature importances:")

for name, imp in sorted(
    zip(FEATURE_NAMES, perm.importances_mean),
    key=lambda x: -x[1]
):
    print(f"{name:28s} {imp:.4f}")

Feature importances:
pair_freq_with_revealed      0.2180
position_freq_max            0.0560
position_freq_avg            0.0366
global_freq                  0.0348
progress                     0.0143
norm_word_length             0.0065
length_freq                  0.0062
pair_freq_with_wrong         0.0036
blank_ratio                  0.0032
guessed_ratio                -0.0004



# Ranking evaluation: does this actually help win Hangman -

## Letter suggestion -

In [29]:
import random
import string
import numpy as np

ALPHABET = list(string.ascii_lowercase)

from features import make_features


def suggest_letters(word_length, pattern, guessed_letters, top_k=5):
    remaining = [
        c for c in ALPHABET
        if c not in guessed_letters
    ]

    if not remaining:
        return []

    feats = np.array([
        make_features(
            word_length,
            pattern,
            guessed_letters,
            c,
            stats
        )
        for c in remaining
    ])

    probs = clf.predict_proba(feats)[:, 1]

    ranked = sorted(
        zip(remaining, probs),
        key=lambda x: -x[1]
    )

    return ranked[:top_k]

## Evaluate the model - 

In [30]:
from features import simulate_game_state

rng = random.Random(7)

holdout_words = rng.sample(vocab, 3000)

top1_hits = 0
top3_hits = 0
total = 0
baseline_top1_hits = 0

letter_freq_order = sorted(
    ALPHABET,
    key=lambda c: -stats["global_freq"][c]
)

for word in holdout_words:

    pattern, guessed = simulate_game_state(word, rng)

    if "_" not in pattern:
        continue

    ranked = suggest_letters(
        len(word),
        pattern,
        guessed,
        top_k=3
    )

    if not ranked:
        continue

    total += 1

    top_letters = [c for c, _ in ranked]

    if top_letters[0] in word:
        top1_hits += 1

    if any(c in word for c in top_letters):
        top3_hits += 1

    baseline_pick = next(
        (c for c in letter_freq_order if c not in guessed),
        None
    )

    if baseline_pick and baseline_pick in word:
        baseline_top1_hits += 1

print(f"Evaluated on {total} mid-game states\n")

print(f"ML model top-1 hit rate: {top1_hits / total:.3f}")
print(f"ML model top-3 hit rate: {top3_hits / total:.3f}")
print(f"Frequency baseline: {baseline_top1_hits / total:.3f}")

Evaluated on 3000 mid-game states

ML model top-1 hit rate: 0.554
ML model top-3 hit rate: 0.858
Frequency baseline: 0.502


# Save artifacts for deployment -

In [31]:
import pickle
import os

with open("artifacts/model.pkl", "wb") as f:
    pickle.dump(clf, f)

print("Saved files:")

for fname in [
    "wordnet_corpus.csv",
    "artifacts/stats.pkl",
    "artifacts/model.pkl"
]:
    print(f"- {fname} ({os.path.getsize(fname):,} bytes)")

Saved files:
- wordnet_corpus.csv (969,822 bytes)
- artifacts/stats.pkl (45,361 bytes)
- artifacts/model.pkl (722,093 bytes)



## 8. Play a full game inline, powered by the trained model

A quick end-to-end demo: pick a random secret word, and let the trained
model choose every guess (no `random.choice()` involved) until it wins or
runs out of guesses.


In [32]:
def play_demo_game(secret=None, max_wrong=6, verbose=True):

    if secret is None:
        secret = random.choice(vocab)

    pattern = ["_"] * len(secret)
    guessed = set()
    wrong = 0

    if verbose:
        print("Secret word length:", len(secret))
        print(" ".join(pattern))

    while wrong < max_wrong and "_" in pattern:

        ranked = suggest_letters(
            len(secret),
            pattern,
            guessed,
            top_k=5
        )

        if not ranked:
            break

        best_letter = ranked[0][0]
        guessed.add(best_letter)

        if verbose:
            top5 = ", ".join(
                f"{c.upper()}({p:.2f})"
                for c, p in ranked
            )
            print(
                f"AI guesses: {best_letter.upper()} | top-5: {top5}"
            )

        if best_letter in secret:

            for i, ch in enumerate(secret):
                if ch == best_letter:
                    pattern[i] = best_letter
        else:
            wrong += 1

        if verbose:
            print(
                " ".join(pattern).upper(),
                f"(wrong guesses: {wrong}/{max_wrong})"
            )

    won = "_" not in pattern

    if verbose:
        print(
            "\nRESULT:",
            "WON 🎉" if won else "LOST 💀",
            "-> secret was",
            secret.upper()
        )

    return won


random.seed(1)
play_demo_game()

Secret word length: 14
_ _ _ _ _ _ _ _ _ _ _ _ _ _
AI guesses: I | top-5: I(0.90), E(0.86), A(0.83), O(0.82), N(0.81)
_ _ _ _ _ _ _ _ _ I I _ _ _ (wrong guesses: 0/6)
AI guesses: E | top-5: E(0.86), N(0.85), O(0.81), T(0.79), A(0.77)
_ _ _ _ E _ _ _ _ I I _ _ E (wrong guesses: 0/6)
AI guesses: N | top-5: N(0.83), O(0.81), R(0.80), T(0.80), S(0.78)
_ _ _ _ E N _ _ N I I _ _ E (wrong guesses: 0/6)
AI guesses: O | top-5: O(0.81), T(0.80), S(0.78), A(0.78), R(0.76)
_ _ _ _ E N _ O N I I _ _ E (wrong guesses: 0/6)
AI guesses: T | top-5: T(0.80), R(0.79), S(0.79), A(0.74), C(0.72)
_ _ _ _ E N T O N I I _ _ E (wrong guesses: 0/6)
AI guesses: S | top-5: S(0.80), R(0.77), A(0.76), C(0.72), L(0.66)
_ _ _ _ E N T O N I I _ _ E (wrong guesses: 1/6)
AI guesses: R | top-5: R(0.79), A(0.77), C(0.72), L(0.68), D(0.54)
_ _ _ _ E N T O N I I _ _ E (wrong guesses: 2/6)
AI guesses: A | top-5: A(0.77), C(0.72), L(0.69), P(0.55), D(0.54)
_ A _ _ E N T O N I I _ A E (wrong guesses: 2/6)
AI guesses: C | top-5

False

In [33]:
random.seed(123)

trials = 300
wins = sum(
    play_demo_game(verbose=False)
    for _ in range(trials)
)

print(f"AI win rate: {wins / trials:.1%}")

AI win rate: 22.7%
